# 01 — Preuve de concept : embeddings CNN + recherche par similarité

Objectif (cadrage, §8) : charger un **ResNet50** pré-entraîné sur ImageNet, extraire les embeddings de **1 000 images** et tester une première recherche par **similarité cosinus**.

À exécuter sur Google Colab / Kaggle (GPU) ou en local (CPU, plus lent).

In [ ]:
# Sur Colab :
# !pip install -q kaggle
# !kaggle datasets download -d paramaggarwal/fashion-product-images-small -p data --unzip
from pathlib import Path

DATA_DIR = Path('../data/fashion-product-images-small')  # styles.csv + images/<id>.jpg
N_IMAGES = 1000

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_DIR / 'styles.csv', on_bad_lines='skip')
df = df[df['masterCategory'].isin(['Apparel', 'Footwear', 'Accessories'])]
df['image_path'] = df['id'].map(lambda i: DATA_DIR / 'images' / f'{i}.jpg')
df = df[df['image_path'].map(Path.exists)].reset_index(drop=True)
print(len(df), 'images')
df['masterCategory'].value_counts()

In [ ]:
# Exploration : quelques images par catégorie
import matplotlib.pyplot as plt
from PIL import Image

for cat, group in df.groupby('masterCategory'):
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
    for ax, (_, row) in zip(axes, group.sample(8, random_state=0).iterrows()):
        ax.imshow(Image.open(row['image_path'])); ax.set_title(row['articleType'], fontsize=8); ax.axis('off')
    fig.suptitle(cat); plt.show()

In [ ]:
sample = df.sample(n=min(N_IMAGES, len(df)), random_state=0).reset_index(drop=True)

import torch
from torch import nn
from torchvision import models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
model.fc = nn.Identity()          # on retire la couche de classification -> sortie 2048-d
model = model.eval().to(device)
preprocess = weights.transforms() # resize 256, crop 224, normalisation ImageNet

In [ ]:
import numpy as np

embeddings = []
with torch.inference_mode():
    for start in range(0, len(sample), 64):
        paths = sample['image_path'][start:start + 64]
        batch = torch.stack([preprocess(Image.open(p).convert('RGB')) for p in paths]).to(device)
        feats = nn.functional.normalize(model(batch), p=2, dim=1)  # norme L2
        embeddings.append(feats.cpu().numpy())
emb = np.concatenate(embeddings)
emb.shape

In [ ]:
# Vecteurs normalisés : similarité cosinus = produit scalaire
def search(q, k=8):
    scores = emb @ emb[q]
    idx = np.argsort(-scores)[1:k + 1]  # [0] = l'image elle-même
    return idx, scores[idx]

def show(q, k=8):
    idx, scores = search(q, k)
    fig, axes = plt.subplots(1, k + 1, figsize=(2 * (k + 1), 3))
    axes[0].imshow(Image.open(sample.loc[q, 'image_path'])); axes[0].set_title('REQUÊTE: ' + sample.loc[q, 'articleType'], fontsize=8)
    for ax, i, s in zip(axes[1:], idx, scores):
        ax.imshow(Image.open(sample.loc[i, 'image_path'])); ax.set_title(f"{sample.loc[i, 'articleType']} ({s:.2f})", fontsize=8)
    for ax in axes: ax.axis('off')
    plt.show()

for q in [0, 10, 20, 30]:
    show(q)

In [ ]:
# Indicateur rapide : part des 5 voisins ayant le même articleType que la requête
labels = sample['articleType'].to_numpy()
sim = emb @ emb.T
np.fill_diagonal(sim, -np.inf)
top5 = np.argsort(-sim, axis=1)[:, :5]
print('Precision@5 (articleType) :', (labels[top5] == labels[:, None]).mean().round(3))

## Suite

Si les voisins sont visuellement cohérents, l'idée est validée. Prochaines phases : nettoyage et split des données (P1), baseline évaluée (P2), fine-tuning (P3).